In [ ]:
import sys
import numpy
import pandas
import sklearn
print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 2962
data = pd.read_csv("data/heart_disease.csv")
feature_names = [c for c in data.columns if c != "disease"]
X, y = data[feature_names].to_numpy(), data["disease"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED
)
print(data.shape, round(y.mean(), 3), X_train.shape, X_test.shape)
assert data.shape == (297, 14) and not data.isna().any().any()

# Task 2 — Decision Trees
**MOP 3231 · Machine Learning (Python)**  
**SEED = 2962** (last four digits of student ID).  
Dataset: Cleveland complete cases, 297 patients. Coded features are used as numbers, without one-hot encoding. This is a teaching exercise, not clinical evidence.

The first two cells follow the handout. Part 2 uses the test features only for implementation agreement; these results do not select hyperparameters. Part 3 selects depth using training-fold results only; its test predictions are reused in Part 4. Written answers are in English to match the assignment.

## Part 1 — Entropy and information gain by hand
Write $h(p)=-p\log_2p-(1-p)\log_2(1-p)$, with $h(0)=h(1)=0$.
There are six positive and six negative patients, so
$$H(S)=-\frac6{12}\log_2\frac6{12}-\frac6{12}\log_2\frac6{12}=1\text{ bit}.$$
Useful substitutions:
$$h(1/6)=-(1/6)\log_2(1/6)-(5/6)\log_2(5/6)=0.650022,$$
$$h(1/4)=-(1/4)\log_2(1/4)-(3/4)\log_2(3/4)=0.811278,$$
$$h(1/3)=-(1/3)\log_2(1/3)-(2/3)\log_2(2/3)=0.918296.$$
Counts below are (positive, negative); gains are in bits.

| Feature | Feature = 0 | Feature = 1 | Information gain calculation | Gain |
|---|---|---|---|---:|
| age_55_plus | (1,5) | (5,1) | $1-[6/12\,h(1/6)+6/12\,h(5/6)]$ | 0.349978 |
| exercise_angina | (2,6) | (4,0) | $1-[8/12\,h(1/4)+4/12\,h(1)]$ | **0.459148** |
| male | (3,3) | (3,3) | $1-[6/12\,h(1/2)+6/12\,h(1/2)]$ | 0.000000 |
| high_bp | (2,4) | (4,2) | $1-[6/12\,h(1/3)+6/12\,h(2/3)]$ | 0.081704 |

The root is **exercise_angina**. Its feature=1 child is pure; feature=0 contains p5–p12 and has entropy $h(2/8)=0.811278$.

| Remaining feature | Feature = 0 | Feature = 1 | Information gain within impure branch | Gain |
|---|---|---|---|---:|
| age_55_plus | (1,5) | (1,1) | $0.811278-[6/8\,h(1/6)+2/8\,h(1/2)]$ | 0.073761 |
| male | (2,3) | (0,3) | $0.811278-[5/8\,h(2/5)+3/8\,h(0)]$ | **0.204434** |
| high_bp | (1,4) | (1,2) | $0.811278-[5/8\,h(1/5)+3/8\,h(1/3)]$ | 0.015712 |

Here $h(2/5)=-(2/5)\log_2(2/5)-(3/5)\log_2(3/5)=0.970951$ and $h(1/5)=0.721928$ (using the same formula). **male** is chosen: its gain increases from 0 at the root to 0.204434 bits in this branch. Information gain depends on the local class distribution: globally both sex groups are balanced, but among patients without exercise angina all three males are negative while two of five females are positive.

In [ ]:
toy = pd.DataFrame({
    "age_55_plus": [1,1,1,1,1,0,1,0,0,0,0,0],
    "exercise_angina": [1,1,1,1,0,0,0,0,0,0,0,0],
    "male": [1,1,1,0,0,0,1,1,1,0,0,0],
    "high_bp": [1,1,0,1,1,0,0,0,0,1,1,0],
    "disease": [1,1,1,1,1,1,0,0,0,0,0,0],
})

## Part 2 — A decision tree from scratch
The implementation enumerates features and midpoint thresholds in ascending order. A tolerance of $10^{-12}$ treats numerical round-off in theoretically equal gains as a tie, preserving the lower feature index and then lower threshold. Valid zero-gain splits are allowed: the stated stopping rules do not stop solely at zero gain (such splits can reveal interactions).

In [ ]:
def entropy(y):
    """Entropy in bits of an array of integer class labels."""
    y = np.asarray(y)
    if y.size == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return float(-np.sum(p * np.log2(p)))


def information_gain(y, left):
    """Parent entropy minus size-weighted child entropies; left is a boolean mask."""
    y, left = np.asarray(y), np.asarray(left, dtype=bool)
    if y.size == 0:
        return 0.0
    w = left.mean()
    return entropy(y) - w * entropy(y[left]) - (1 - w) * entropy(y[~left])


def best_split(X, y, min_samples_leaf=1):
    """Return (feature, midpoint, gain), with deterministic ties, or None."""
    X, y = np.asarray(X), np.asarray(y)
    if min_samples_leaf < 1:
        raise ValueError("min_samples_leaf must be at least 1")
    best = None
    for j in range(X.shape[1]):
        values = np.unique(X[:, j])
        thresholds = values[:-1] + (values[1:] - values[:-1]) / 2
        for threshold in thresholds:
            left = X[:, j] <= threshold
            n_left = int(left.sum())
            if min(n_left, len(y) - n_left) < min_samples_leaf:
                continue
            gain = information_gain(y, left)
            if best is None or gain > best[2] + 1e-12:
                best = (j, float(threshold), float(gain))
    return best


def build_tree(X, y, max_depth=None, min_samples_leaf=1, depth=0):
    """Recursively grow nested dictionaries; tied majorities use the smaller label."""
    X, y = np.asarray(X), np.asarray(y)
    if len(y) == 0:
        raise ValueError("Cannot train on an empty node")
    labels, counts = np.unique(y, return_counts=True)
    node = {"prediction": int(labels[np.argmax(counts)]), "n_samples": len(y)}
    if len(labels) == 1 or (max_depth is not None and depth >= max_depth):
        return node
    split = best_split(X, y, min_samples_leaf)
    if split is None:
        return node
    j, threshold, gain = split
    left = X[:, j] <= threshold
    node.update(feature=j, threshold=threshold, gain=gain,
                left=build_tree(X[left], y[left], max_depth, min_samples_leaf, depth+1),
                right=build_tree(X[~left], y[~left], max_depth, min_samples_leaf, depth+1))
    return node


def predict_tree(tree, X):
    """Return one predicted integer label for each row of X."""
    predictions = []
    for row in np.asarray(X):
        node = tree
        while "feature" in node:
            node = node["left"] if row[node["feature"]] <= node["threshold"] else node["right"]
        predictions.append(node["prediction"])
    return np.asarray(predictions, dtype=int)

In [ ]:
from IPython.display import display, Markdown
from sklearn.tree import DecisionTreeClassifier

print("Toy parent entropy:", entropy(toy.disease))
root_gains = {f: information_gain(toy.disease.to_numpy(), toy[f].to_numpy() == 0)
              for f in toy.columns[:-1]}
branch = toy[toy.exercise_angina == 0]
branch_gains = {f: information_gain(branch.disease.to_numpy(), branch[f].to_numpy() == 0)
                for f in ["age_55_plus", "male", "high_bp"]}
display(pd.Series(root_gains, name="root gain").to_frame())
display(pd.Series(branch_gains, name="branch gain").to_frame())
assert np.allclose(list(root_gains.values()), [0.3499775784,0.4591479170,0,0.0817041659])
assert np.allclose(list(branch_gains.values()), [0.0737613082,0.2044340029,0.0157121274])
print("All hand calculations reproduced.")

ref = DecisionTreeClassifier(criterion="entropy", max_depth=1, random_state=SEED).fit(X_train,y_train)
split = best_split(X_train,y_train)
print("NumPy root:", feature_names[split[0]], split[1], "gain:", split[2])
print("sklearn root:", feature_names[ref.tree_.feature[0]], ref.tree_.threshold[0])
print("Same feature and threshold:", split[0] == ref.tree_.feature[0] and
      np.isclose(split[1], ref.tree_.threshold[0]))
rows = []
for depth in [1,2,3,4,5,None]:
    own = build_tree(X_train,y_train,max_depth=depth)
    reference = DecisionTreeClassifier(criterion="entropy",max_depth=depth,
                                      random_state=SEED).fit(X_train,y_train)
    agreement = np.mean(predict_tree(own,X_test) == reference.predict(X_test))
    rows.append({"max_depth": str(depth), "agreement": agreement,
                 "agreement_percent": 100*agreement})
agreement_table = pd.DataFrame(rows)
display(agreement_table)
display(Markdown("Any disagreement can arise because scikit-learn permutes candidate features, "
                 "so tied gains need not use our lower-index tie rule; it also uses float32 "
                 "feature values internally. The same root does not guarantee identical deeper branches."))

## Part 3 — How deep should the tree be?
### Selection rule (defined before evaluating the sweep)
Use the **one-standard-error rule**: find the largest mean CV accuracy, calculate its fold standard deviation divided by $\sqrt{5}$, and choose the shallowest depth with mean CV accuracy at least (best mean − that standard error). None is considered more complex than every finite candidate. This uses the spread of the folds to favor a simpler model when scores are similar; folds overlap in training observations, so the standard error is a heuristic rather than a formal confidence interval.

All sweeps use only the 207 training observations and the same five stratified folds. “Leaves” means leaves of a model fitted to all 207 training rows, not an average over folds. CV standard deviations use `ddof=0`, matching the handout. The leaf-size sweep is a separate comparison; it does not replace the depth selected by this rule.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
import matplotlib.pyplot as plt

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
depths = [1,2,3,4,5,6,8,10,None]
rows = []
for depth in depths:
    model = DecisionTreeClassifier(criterion="entropy", max_depth=depth, random_state=SEED)
    scores = cross_validate(model,X_train,y_train,cv=cv,scoring="accuracy",return_train_score=True)
    model.fit(X_train,y_train)
    rows.append({"depth": "None" if depth is None else str(depth),
                 "leaves": model.get_n_leaves(),
                 "train_mean": scores["train_score"].mean(),
                 "cv_mean": scores["test_score"].mean(),
                 "cv_std": scores["test_score"].std()})
depth_table = pd.DataFrame(rows)
display(depth_table.round(6))
best_i = int(depth_table.cv_mean.idxmax())
cutoff = depth_table.loc[best_i,"cv_mean"] - depth_table.loc[best_i,"cv_std"]/np.sqrt(5)
chosen_i = int(depth_table.index[depth_table.cv_mean >= cutoff][0])
chosen_depth = depths[chosen_i]
print(f"Best mean at depth {depths[best_i]}; one-SE cutoff={cutoff:.6f}; chosen depth={chosen_depth}")

fig,ax = plt.subplots(figsize=(9,4.5))
positions = np.arange(len(depths))
ax.plot(positions, depth_table.train_mean, "o-", label="Mean training accuracy")
ax.errorbar(positions, depth_table.cv_mean, yerr=depth_table.cv_std,
            fmt="o-", capsize=4, label="Mean CV accuracy ± 1 fold SD")
ax.axhline(cutoff, color="gray", linestyle=":", label="One-SE cutoff")
ax.axvline(chosen_i, color="green", alpha=.4, linestyle="--", label="Selected depth")
ax.set(xticks=positions,xticklabels=depth_table.depth, xlabel="Maximum depth (None = unlimited)",
       ylabel="Accuracy",ylim=(.5,1.03),title="Tree complexity: five-fold training-only CV")
ax.legend(loc="best",fontsize=9)
ax.grid(alpha=.2)
fig.tight_layout()
plt.show()

selected = depth_table.loc[chosen_i]
neighbors = depth_table.iloc[max(0,chosen_i-1):min(len(depth_table),chosen_i+2)]
display(Markdown(f"The rule selects **depth {chosen_depth}**, the shallowest model above "
                 f"the cutoff {cutoff:.4f}, with CV accuracy {selected.cv_mean:.4f} ± {selected.cv_std:.4f}."))
display(neighbors[["depth","cv_mean","cv_std"]])
unlimited = depth_table.iloc[-1]
display(Markdown(f"At unlimited depth, mean training accuracy {unlimited.train_mean:.4f} "
                 f"exceeds CV accuracy {unlimited.cv_mean:.4f} by "
                 f"{unlimited.train_mean-unlimited.cv_mean:.4f}, evidence of overfitting."))

In [ ]:
rows = []
for minimum in [1,2,5,10,20,40]:
    model = DecisionTreeClassifier(criterion="entropy",min_samples_leaf=minimum,random_state=SEED)
    scores = cross_validate(model,X_train,y_train,cv=cv,scoring="accuracy",return_train_score=True)
    model.fit(X_train,y_train)
    rows.append({"min_samples_leaf":minimum,"leaves":model.get_n_leaves(),
                 "train_mean":scores["train_score"].mean(),"cv_mean":scores["test_score"].mean(),
                 "cv_std":scores["test_score"].std()})
leaf_table = pd.DataFrame(rows)
display(leaf_table.round(6))

Maximum depth caps the number of decisions on every path, while minimum leaf size forbids small terminal groups and may still allow long paths in well-supported regions. Both limit overfitting, but an excessively shallow tree or large minimum leaf size can underfit.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
chosen_tree = DecisionTreeClassifier(criterion="entropy",max_depth=chosen_depth,random_state=SEED)
chosen_tree.fit(X_train,y_train)
tree_test_predictions = chosen_tree.predict(X_test)
tree_test_accuracy = accuracy_score(y_test,tree_test_predictions)
tree_test_se = np.sqrt(tree_test_accuracy*(1-tree_test_accuracy)/len(y_test))
print(f"Selected tree test accuracy: {tree_test_accuracy:.6f}; SE: {tree_test_se:.6f}; n={len(y_test)}")

## Part 4 — Tree or logistic regression?
All models use the same training data and stratified folds. Logistic regression scales inside its pipeline, so each scaler is fitted only on its fold's training rows. Positive class for F1 is disease=1. The selected tree's hold-out predictions from Part 3 are reused rather than selecting a new model after seeing the test results.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

logreg = make_pipeline(StandardScaler(),LogisticRegression(max_iter=5000))
models = {"Selected tree":DecisionTreeClassifier(criterion="entropy",max_depth=chosen_depth,random_state=SEED),
          "Unlimited tree":DecisionTreeClassifier(criterion="entropy",random_state=SEED),
          "Logistic regression":logreg}
rows = []
for name,model in models.items():
    scores = cross_validate(model,X_train,y_train,cv=cv,scoring=("accuracy","f1"))
    rows.append({"model":name,"cv_accuracy_mean":scores["test_accuracy"].mean(),
                 "cv_accuracy_std":scores["test_accuracy"].std(),"cv_f1_mean":scores["test_f1"].mean()})
comparison = pd.DataFrame(rows).set_index("model")
display(comparison.round(6))

scaler = StandardScaler().fit(X_train)
scaled_tree = DecisionTreeClassifier(criterion="entropy",max_depth=chosen_depth,random_state=SEED)
scaled_tree.fit(scaler.transform(X_train),y_train)
scaled_predictions = scaled_tree.predict(scaler.transform(X_test))
scaling_agreement = np.mean(tree_test_predictions == scaled_predictions)
print(f"Raw/scaled tree prediction agreement: {scaling_agreement:.6f} ({100*scaling_agreement:.2f}%)")

logreg.fit(X_train,y_train)
logreg_test_predictions = logreg.predict(X_test)
holdout_rows=[]
for name,predictions in [("Selected tree",tree_test_predictions),("Logistic regression",logreg_test_predictions)]:
    accuracy = accuracy_score(y_test,predictions)
    holdout_rows.append({"model":name,"accuracy":accuracy,"f1":f1_score(y_test,predictions),
                         "accuracy_se":np.sqrt(accuracy*(1-accuracy)/len(y_test))})
    print(name,"confusion matrix (rows=true, columns=predicted; labels=[0,1]):")
    display(pd.DataFrame(confusion_matrix(y_test,predictions,labels=[0,1]),
                         index=["True 0","True 1"],columns=["Predicted 0","Predicted 1"]))
holdout = pd.DataFrame(holdout_rows).set_index("model")
display(holdout.round(6))

StandardScaler applies a positive affine transformation to each nonconstant feature, preserving order and translating midpoint thresholds accordingly; trees do not need scaling. Predictions should therefore agree, except for possible floating-point or split-tie effects near thresholds; the measured agreement is reported above.